# NSR Hyperparameter Search on Multiple KGs

This notebook demonstrates hyperparameter search for `NSR/NSR_Simplified.py` on several standard KG benchmarks.

Features:
- Caches the learned C layer so that W_RR + fallback hyperparameters can be tuned without relearning C.
- Uses Optuna to search `W_RR_learning_algorithm`, `T_thresh`, `alpha`, `agg_mode`, and the two fallbacks (`tail_fallback_weight`, `concepts_fallback_weight`).
- Reports MRR and Hits@1/3/10 on the test set.


## 1. Evaluation utilities


In [ ]:
import random
import time
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def evaluate_ranking_filtered(score_fn, eval_triples, num_entities, known_triples, seed=42):
    rng = random.Random(seed)
    known_hr2t = defaultdict(set)
    for h, r, t in known_triples:
        known_hr2t[(h, r)].add(t)

    ranks = []
    hits_at_1 = hits_at_3 = hits_at_10 = 0
    scores_arr = np.zeros(num_entities, dtype=np.float64)

    for h, r, t in eval_triples:
        scores_dict = score_fn(h, r)
        s_true = scores_dict.get(t, 0.0)
        scores_arr[:] = 0.0
        for e, s in scores_dict.items():
            scores_arr[int(e)] = float(s)
        for e in known_hr2t[(h, r)]:
            if e != t:
                scores_arr[int(e)] = -np.inf
        noise = np.fromiter((rng.uniform(0.0, 1e-6) for _ in range(num_entities)),
                            dtype=np.float64, countnegativebehavior=num_entities)
        scores_arr += noise

        higher = scores_arr > scores_arr[t] + 1e-12
        tie = (np.abs(scores_arr - scores_arr[t]) <= 1e-12) & (np.arange(num_entities) < t)
        tie[t] = False
        rank = 1 + int(np.count_nonzero(higher)) + int(np.count_nonzero(tie))
        ranks.append(rank)
        if rank <= 1: hits_at_1 += 1
        if rank <= 3: hits_at_3 += 1
        if rank <= 10: hits_at_10 += 1

    n = len(eval_triples)
    ranks = np.asarray(ranks, dtype=float)
    return {
        "MR": float(ranks.mean()),
        "MRR": float((1.0 / ranks).mean()),
        "Hits@1": 100.0 * hits_at_1 / n,
        "Hits@3": 100.0 * hits_at_3 / n,
        "Hits@10": 100.0 * hits_at_10 / n,
    }


def print_metrics(name, met):
    print(f"{name}: MR={met['MR']:.4f}, MRR={met['MRR']:.4f}, "
          f"Hits@1={met['Hits@1']:.2f}%, Hits@3={met['Hits@3']:.2f}%, "
          f"Hits@10={met['Hits@10']:.2f}%")


## 2. Load model and dataset helpers

The model file is loaded from `NSR/NSR_Simplified.py` (project-relative path).


In [ ]:
import sys
import importlib.util
from dataclasses import fields, is_dataclass


def load_module_from_file(name, path):
    path = Path(path).resolve()
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


nsr_module = load_module_from_file("nsr_simplified", "NSR/NSR_Simplified.py")
NSRConfig = nsr_module.NSRConfig
NSRDynamicERZC = nsr_module.NSRDynamicERZC


def load_kinship_1990_splits(path, seed=42):
    import re
    entity_to_id = {}
    relation_to_id = {}
    pattern = re.compile(r"([^\(]+)\(([^,]+),\s*([^\)]+)\)")

    def get_id(d, name):
        if name not in d:
            d[name] = len(d)
        return d[name]

    def parse(p):
        triples = []
        for line in Path(p).read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            m = pattern.match(line)
            if not m:
                parts = line.replace("\t", " ").split()
                if len(parts) != 3:
                    continue
                h, r, t = parts
            else:
                r, h, t = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
            triples.append((get_id(entity_to_id, h), get_id(relation_to_id, r), get_id(entity_to_id, t)))
        return triples

    data_dir = Path(path)
    train = parse(data_dir / "train.data")
    valid = parse(data_dir / "valid.data")
    test = parse(data_dir / "test.data")
    return {
        "train": train, "valid": valid, "test": test,
        "N": len(entity_to_id), "M": len(relation_to_id),
    }


## 3. Cached-C hyperparameter search

For each dataset we:
1. Encode triples and learn the C layer once with `_learn_C_layer_matrix()`.
2. Save the model to `NSR/Results/tune_wrr/cache/{dataset}_c_layer.npz`.
3. Run Optuna over W_RR + inference + fallback hyperparameters.
4. Evaluate the best valid configuration on the test set.

Toggle `USE_BEST_HYPERPARAMS` to skip search and use stored best parameters.


In [ ]:
from pykeen.datasets import Kinships, FB15k237, WN18RR, YAGO310, Nations


def pykeen_to_dict(ds):
    return {
        "train": ds.training.mapped_triples.detach().cpu().numpy().tolist(),
        "valid": ds.validation.mapped_triples.detach().cpu().numpy().tolist(),
        "test": ds.testing.mapped_triples.detach().cpu().numpy().tolist(),
        "N": ds.num_entities,
        "M": ds.num_relations,
    }


def make_cfg(params, c_params):
    # Build NSRConfig. C-layer params come from the cached config.
    merged = dict(c_params)
    merged.update(params)
    # Keep C-layer params identical to cached model.
    for k in ("C_activity_threshold", "C_rule_threshold", "C_min_support"):
        merged[k] = c_params[k]
    kwargs = {k: v for k, v in merged.items() if k in {f.name for f in fields(NSRConfig)}}
    return NSRConfig(**kwargs)


def get_search_space(trial):
    algo = trial.suggest_categorical("W_RR_learning_algorithm", ["cooccurrence", "online"])
    params = {
        "W_RR_learning_algorithm": algo,
        "T_thresh": trial.suggest_float("T_thresh", 0.0, 1.0),
        "W_RR_normalize": trial.suggest_categorical("W_RR_normalize", ["sigmoid", "max", "none"]),
        "W_RR_sigmoid_k": trial.suggest_float("W_RR_sigmoid_k", 0.1, 5.0),
        "agg_mode": trial.suggest_categorical("agg_mode", ["max", "add"]),
        "alpha": trial.suggest_float("alpha", 0.0, 2.0),
        "tail_fallback_weight": trial.suggest_float("tail_fallback_weight", 0.0, 0.05),
        "concepts_fallback_weight": trial.suggest_float("concepts_fallback_weight", 0.0, 1.0),
        "concepts_fallback_k": trial.suggest_int("concepts_fallback_k", 1, 10),
    }
    if algo == "online":
        params["delta_RR"] = trial.suggest_float("delta_RR", 0.05, 1.0)
        params["mu_RR"] = trial.suggest_float("mu_RR", 0.0, 0.2)
    return params


_RESULTS = {}


def tune_and_eval(name, data, n_trials=20, use_cache=True, seed=42):
    cache_dir = Path("NSR/Results/tune_wrr/cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{name.lower()}_c_layer.npz"

    # C-layer config: matrix algorithm, always sparse storage.
    c_params = {
        "T_thresh": 0.0,
        "W_RR_sigmoid_k": 1.0,
        "C_activity_threshold": 0.5,
        "C_rule_threshold": 0.1,
        "C_min_support": 2,
        "C_learning_algorithm": "matrix",
        "C_dense_memory_limit_bytes": 1,
        "tail_fallback_weight": 0.0,
        "concepts_fallback_weight": 0.0,
        "concepts_fallback_k": 3,
    }

    if cache_path.exists() and use_cache:
        print(f"[{name}] Loading cached C layer from {cache_path}")
        model = NSRDynamicERZC.load(str(cache_path))
        c_time = 0.0
    else:
        print(f"[{name}] Learning C layer...")
        cfg = make_cfg({}, c_params)
        model = NSRDynamicERZC(data["N"], data["M"], cfg)
        model.encode_triples(data["train"])
        t0 = time.perf_counter()
        model.learn_C_layer(seed=seed, reset=True, verbose=False)
        c_time = time.perf_counter() - t0
        model.save(str(cache_path))
        print(f"[{name}] C layer done in {c_time:.2f}s, "
              f"active_rules={model.C_learning_stats.get('active_rules', -1)}")

    # Fix C-layer params to the cached config.
    for k in ("C_activity_threshold", "C_rule_threshold", "C_min_support"):
        c_params[k] = getattr(model.cfg, k)

    all_triples = data["train"] + data["valid"] + data["test"]
    known = defaultdict(set)
    for h, r, t in all_triples:
        known[(h, r)].add(t)

    def objective(trial):
        params = get_search_space(trial)
        m = NSRDynamicERZC.load(str(cache_path))
        m.cfg = make_cfg(params, c_params)
        m.learn_WRR_hebbian(
            data["train"], n_passes=1,
            normalize=params["W_RR_normalize"],
            seed=seed, reset=True, verbose=False,
        )
        met = evaluate_ranking_filtered(
            lambda h, r: m.query(h, r, use_C_layer=True,
                                 agg_mode=params["agg_mode"],
                                 alpha=params["alpha"]),
            data["valid"], data["N"], all_triples, seed=seed,
        )
        return met["MRR"]

    print(f"[{name}] Running {n_trials} Optuna trials...")
    study = optuna.create_study(direction="maximize")
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=n_trials)
    search_time = time.perf_counter() - t0

    best = study.best_params
    print(f"[{name}] Best valid MRR={study.best_value:.4f}, params={best}")

    # Test evaluation.
    m = NSRDynamicERZC.load(str(cache_path))
    m.cfg = make_cfg(best, c_params)
    t0 = time.perf_counter()
    m.learn_WRR_hebbian(
        data["train"], n_passes=1,
        normalize=best["W_RR_normalize"],
        seed=seed, reset=True, verbose=False,
    )
    wrr_time = time.perf_counter() - t0

    no_c = evaluate_ranking_filtered(
        lambda h, r: m.query(h, r, use_C_layer=False),
        data["test"], data["N"], all_triples, seed=seed,
    )
    with_c = evaluate_ranking_filtered(
        lambda h, r: m.query(h, r, use_C_layer=True,
                              agg_mode=best["agg_mode"],
                              alpha=best["alpha"]),
        data["test"], data["N"], all_triples, seed=seed,
    )

    print_metrics(f"[{name}] no-C", no_c)
    print_metrics(f"[{name}] with-C", with_c)

    _RESULTS[name] = {
        "best_valid_mrr": study.best_value,
        "best_params": best,
        "c_train_sec": c_time,
        "wrr_train_sec": wrr_time,
        "search_sec": search_time,
        "no_C": no_c,
        "with_C": with_c,
    }
    return _RESULTS[name]


### WN18RR


In [ ]:
import ssl, os
os.environ['PYTHONHTTPSVERIFY'] = '0'
ssl._create_default_https_context = ssl._create_unverified_context

ds = pykeen_to_dict(WN18RR())
print(f"[WN18RR] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("WN18RR", ds, n_trials=20)


### FB15K-237


In [ ]:
ds = pykeen_to_dict(FB15k237())
print(f"[FB15K-237] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("FB15K-237", ds, n_trials=20)


### YAGO3-10


In [ ]:
ds = pykeen_to_dict(YAGO310())
print(f"[YAGO3-10] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("YAGO3-10", ds, n_trials=15)


### KINSHIP


In [ ]:
ds = pykeen_to_dict(Kinships())
print(f"[KINSHIP] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("KINSHIP", ds, n_trials=20)


### KINSHIP_1990_EXTENDED


In [ ]:
ds = load_kinship_1990_splits("KINSHIP_1990")
print(f"[KINSHIP_1990] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("KINSHIP_1990_EXTENDED", ds, n_trials=20)


### NATIONS


In [ ]:
ds = pykeen_to_dict(Nations())
print(f"[NATIONS] N={ds['N']}, M={ds['M']}, train={len(ds['train'])}, "
      f"valid={len(ds['valid'])}, test={len(ds['test'])}")
tune_and_eval("NATIONS", ds, n_trials=30)


## 4. Summary


In [ ]:
if _RESULTS:
    rows = []
    for name, r in _RESULTS.items():
        rows.append({
            "Dataset": name,
            "C train (s)": f"{r['c_train_sec']:.2f}",
            "W_RR train (s)": f"{r['wrr_train_sec']:.4f}",
            "Search (s)": f"{r['search_sec']:.2f}",
            "Valid MRR": f"{r['best_valid_mrr']:.4f}",
            "Test no-C MRR": f"{r['no_C']['MRR']:.4f}",
            "Test with-C MRR": f"{r['with_C']['MRR']:.4f}",
            "Hits@1": f"{r['with_C']['Hits@1']:.2f}%",
            "Hits@3": f"{r['with_C']['Hits@3']:.2f}%",
            "Hits@10": f"{r['with_C']['Hits@10']:.2f}%",
        })
    display(pd.DataFrame(rows))
else:
    print("No results yet. Run the dataset cells above first.")
